In [ ]:
# 프로젝트 경로 설정
from pathlib import Path
import os
import sys

project_root = next(
    (folder for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / "project_paths.py").is_file()),
    None,
)
if project_root is None:
    raise RuntimeError("프로젝트 폴더에서 노트북을 실행하세요.")
if str(project_root) in sys.path:
    sys.path.remove(str(project_root))
sys.path.insert(0, str(project_root))
sys.modules.pop("project_paths", None)
from project_paths import DATA_ROOT, RAW_DIR, PROCESSED_DIR, OUTPUT_DIR, PATIENT_MAPPING_PATH, deidentified_path

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import os
import glob
import pandas as pd
from datetime import timedelta

# 경로 설정
table_dir = str(PROCESSED_DIR / 'AEGIS-Order-CIV2/de')

# 환자별 노이즈 정보
patient_mapping_df = pd.read_csv(str(PATIENT_MAPPING_PATH), sep=",", quotechar ='"', lineterminator = '\n',dtype = {'pt_id': str})

# 노이즈 매핑 딕셔너리
patient_mapping_df['new_pt_id'] = patient_mapping_df['new_pt_id'].astype(str)

patient_id_map = patient_mapping_df.set_index('pt_id')['new_pt_id'].to_dict()
date_offset_map = patient_mapping_df.set_index('new_pt_id')['date_noise'].astype(int).to_dict()
time_offset_map = patient_mapping_df.set_index('new_pt_id')['time_noise'].astype(int).to_dict()

# 대상 파일 수집
csv_files = glob.glob(os.path.join(table_dir, '**/*.csv'), recursive = True)
print(f"[INFO] 총 처리 대상 파일 수 : {len(csv_files)}")


In [ ]:
# 1. pt_id 치환
print("\n[INFO] 1. pt_id 치환 시작")
for file in csv_files:
    try:
        civ2_orders_df = pd.read_csv(file, dtype={'pt_id':str},sep=',', quotechar='"', lineterminator='\n')

        if 'pt_id' in civ2_orders_df.columns:
            civ2_orders_df['pt_id'] = civ2_orders_df['pt_id'].map(patient_id_map).fillna(civ2_orders_df['pt_id'])

        print(civ2_orders_df.head(3))

        civ2_orders_df.to_csv(file, index=False, encoding = 'utf-8')
        print(f"[INFO] pt_id 저장 완료: {file}")
    except Exception as e:
        print(f"[ERROR] pt_id 오류: {file} → {e}")


In [ ]:
# 2. hos_dt ±7일 ±30분
print("\n[INFO] 2. hos_dt 노이즈 적용")
for file in csv_files:
    try:
        civ2_orders_df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')

        if 'hos_dt' in civ2_orders_df.columns:
            civ2_orders_df['hos_dt'] = pd.to_datetime(civ2_orders_df['hos_dt'], errors='coerce')
            civ2_orders_df['hos_dt'] = civ2_orders_df['hos_dt'] + civ2_orders_df['pt_id'].map(lambda x: timedelta(days=date_offset_map.get(x, 0), minutes=time_offset_map.get(x, 0)))

            print(civ2_orders_df.head(3))

            civ2_orders_df.to_csv(file, index=False, encoding = 'utf-8')
            print(f"[INFO] hos_dt 저장 완료: {file}")
    except Exception as e:
        print(f"[ERROR] hos_dt 오류: {file} → {e}")


In [ ]:
# order_date 처리

for file in csv_files:

    try:

        civ2_orders_df = pd.read_csv(file, dtype={'pt_id': str}, sep=",", quotechar ='"', lineterminator = '\n')


        if 'order_date' in civ2_orders_df.columns:

            civ2_orders_df['order_date'] = pd.to_datetime(civ2_orders_df['order_date'], errors = 'coerce')



            civ2_orders_df['order_date'] = civ2_orders_df['order_date'] \
                + pd.to_timedelta(civ2_orders_df['pt_id'].map(date_offset_map).fillna(0).astype(int), unit = "D")


            print(civ2_orders_df.head(3))



            civ2_orders_df.to_csv(file, index = False, encoding = 'utf-8')

            print(f"[INFO] order_date 완료: {file}")





    except Exception as e:

        print(f"[ERROR] order_date 오류: {file} -> {e}")


In [ ]:
# civ_dt ±7일 ±30분
print("\n[INFO] civ_dt 노이즈 적용")
for file in csv_files:
    try:
        civ2_orders_df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')

        if 'civ_dt' in civ2_orders_df.columns:
            civ2_orders_df['civ_dt'] = pd.to_datetime(civ2_orders_df['civ_dt'], errors='coerce')
            civ2_orders_df['civ_dt'] = civ2_orders_df['civ_dt'] + civ2_orders_df['pt_id'].map(lambda x: timedelta(days=date_offset_map.get(x, 0), minutes=time_offset_map.get(x, 0)))

            print(civ2_orders_df.head(3))

            civ2_orders_df.to_csv(file, index=False, encoding = 'utf-8')
            print(f"[INFO] civ_dt 저장 완료: {file}")
    except Exception as e:
        print(f"[ERROR] civ_dt 오류: {file} → {e}")


In [ ]:
# civ_med_weight_value 처리
import numpy as np

for file in csv_files:
    try:
        civ2_orders_df = pd.read_csv(file, dtype={'pt_id': str}, sep=",", quotechar='"', lineterminator='\n')

        if 'civ_med_weight_value' in civ2_orders_df.columns:
            civ2_orders_df['civ_med_weight_value'] = pd.to_numeric(civ2_orders_df['civ_med_weight_value'], errors='coerce')

            # 2. 상하위 1% 경계값 계산
            lower_bound = civ2_orders_df['civ_med_weight_value'].quantile(0.01)
            upper_bound = civ2_orders_df['civ_med_weight_value'].quantile(0.99)

            # 3. 상하위 1% 바깥값은 경계값으로 대체 (clip)
            civ2_orders_df['civ_med_weight_value'] = civ2_orders_df['civ_med_weight_value'].clip(lower=lower_bound, upper=upper_bound)

            # 소수점 제거
            civ2_orders_df['civ_med_weight_value'] = civ2_orders_df['civ_med_weight_value'].apply(
            lambda x: np.floor(x) if pd.notna(x) else np.nan
            )

            print(civ2_orders_df['civ_med_weight_value'])
            # 저장
            civ2_orders_df.to_csv(file, index=False, encoding='utf-8')
            print(f"[INFO] civ_med_weight_value 처리 완료: {file}")

    except Exception as e:
        print(f"[ERROR] civ_med_weight_value 오류: {file} → {e}")


In [ ]:
# order_nursing_dt ±7일 ±30분
print("\n[INFO] order_nursing_dt 노이즈 적용")
for file in csv_files:
    try:
        civ2_orders_df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')

        if 'order_nursing_dt' in civ2_orders_df.columns:
            civ2_orders_df['order_nursing_dt'] = pd.to_datetime(civ2_orders_df['order_nursing_dt'], errors='coerce')
            civ2_orders_df['order_nursing_dt'] = civ2_orders_df['order_nursing_dt'] + civ2_orders_df['pt_id'].map(lambda x: timedelta(days=date_offset_map.get(x, 0), minutes=time_offset_map.get(x, 0)))

            print(civ2_orders_df.head(3))

            civ2_orders_df.to_csv(file, index=False, encoding = 'utf-8')
            print(f"[INFO] order_nursing_dt 저장 완료: {file}")
    except Exception as e:
        print(f"[ERROR] order_nursing_dt 오류: {file} → {e}")


In [ ]:
# 250924 추가
# medicine_order_date ±7일

for file in csv_files:

    try:

        civ2_orders_df = pd.read_csv(file, dtype={'pt_id': str}, sep=",", quotechar ='"', lineterminator = '\n')


        if 'medicine_order_date' in civ2_orders_df.columns:

            civ2_orders_df['medicine_order_date'] = pd.to_datetime(civ2_orders_df['medicine_order_date'], errors = 'coerce')



            civ2_orders_df['medicine_order_date'] = civ2_orders_df['medicine_order_date'] \
                + pd.to_timedelta(civ2_orders_df['pt_id'].map(date_offset_map).fillna(0).astype(int), unit = "D")


            print(civ2_orders_df.head(3))



            civ2_orders_df.to_csv(file, index = False, encoding = 'utf-8-sig')

            print(f"[INFO] order_date 완료: {file}")





    except Exception as e:

        print(f"[ERROR] order_date 오류: {file} -> {e}")
